In [96]:
import sys
import json
import os
sys.path.append( '/home/lodes/uni/3.semester/project/LLaVA-3D/open3dsg' )
import torch
import numpy as np
from const import CONF_PATH_R3SCAN_RAW, CONF_PATH_R3SCAN_PROCESSED
from open_dataset import Open2D3DSGDataset
import re
from graphviz import Digraph
import matplotlib.colors as mcolors
colors = list(mcolors.TABLEAU_COLORS.keys())
node_color_list = list(mcolors.TABLEAU_COLORS.values())
obj_class_dict = [line.rstrip() for line in open(os.path.join(CONF_PATH_R3SCAN_RAW, "classes.txt"), "r").readlines()]

In [97]:
def load_scan(base_path, file_path):
            return json.load(open(os.path.join(base_path, file_path)))["scans"]

def get_queries(data_dict):
    obj_class_dict = [line.rstrip() for line in open(os.path.join(CONF_PATH_R3SCAN_RAW, "classes.txt"), "r").readlines()]
    obj_count = data_dict['objects_count'].item()
    rel_count = int(data_dict["predicate_count"].item())
    objects_gt = data_dict['objects_cat']
    edges = data_dict['edges'][:rel_count]
    object_edges = np.array(objects_gt[:obj_count][edges], dtype=np.int32)
    object_edges = np.array(obj_class_dict)[object_edges]
    object_edges[object_edges == 'socket'] = 'wall'
    queries = [
    f"Describe the relationship between the {o[0]} and the {'other ' if o[0]==o[1] else ''}{o[1]}. Start the response with: the {o[0]}" if o[0] != o[1]
    else f"Describe the relationship between the {o[0]} and the {'other ' if o[0]==o[1] else ''}{o[1]}. Start the response with: the {o[0]}"
    for o in object_edges]

    return queries


def vis_graph_scannet_clip(scan_id, objects_gt, predicates, edges, object_ids, filename='graph'):
    dot = Digraph(comment='The Scene Graph')
    dot.attr(rankdir='TB')

    dot.attr(label=scan_id)
    dot.attr('node', shape='oval', fontname='Sans')
    a = scan_id
    a, b = '-'.join(a.split('-')[:-1]), a.split('-')[-1]
    g_colors = node_color_list  # {o['id']: o['ply_color'] for o in self.scene_graphs_val[a+'_'+b]['objects']}
    for index in range(len(objects_gt)):
        id = str(index)
        dot.attr('node', fillcolor=g_colors[index], style='filled')
        #pred = obj_class_dict[objects_gt[index]]
        pred = objects_gt[index]
        pred = pred if pred != 'socket' else 'wall'
        #dot.node(id, pred+f" [{objects_gt[index]}] "+'-'+str(object_ids[index].item()))
        dot.node(id, pred)

    edges
    dot.attr('edge', fontname='Sans', color='black', style='filled')
    for i, edge in enumerate(edges):
        s, o = edge[:2]
        p_s = predicates[i]
        if np.array([none_p in p_s for none_p in ['and',  'unrelated', 'not', 'none']]).any():
            # if p_s in ['', 'and', ' ', 'unrelated', 'not', 'none']:
            continue
        dot.edge(str(s.item()), str(o.item()), p_s)
    dot.render(filename, format="png", cleanup=True)

In [98]:
with open('./blip_relationships/results.json') as results:
    relationships = json.load(results)
    
scan_id = '754e884c-ea24-2175-8b34-cead19d4198d'
D3SSG = load_scan(CONF_PATH_R3SCAN_RAW, "relationships_train.json")
for r in D3SSG:
    if r['scan'] == scan_id:
        D3SSG = [r]
dataset = Open2D3DSGDataset(
    relationships_R3SCAN=D3SSG,
    relationships_scannet=None,
    openseg=False,
    img_dim=224,
    rel_img_dim=224,
    top_k_frames=5,
    scales=3,
    mini=False,
    load_features=None,
    blip=True,
    llava=False,
    half=False,
    max_objects=9,
    max_rels=72
)
data_dict = dataset[0]

  0%|          | 0/1 [00:00<?, ?it/s]

In [99]:
obj_count = data_dict['objects_count'].item()
rel_count = int(data_dict["predicate_count"].item())
objects_gt = data_dict['objects_cat']
edges = data_dict['edges'][:rel_count]
object_edges = np.array(objects_gt[:obj_count][edges], dtype=np.int32)
object_edges = np.array(obj_class_dict)[object_edges]
object_edges[object_edges == 'socket'] = 'wall'
qs = [
f"Describe the relationship between the {o[0]} and the {'other ' if o[0]==o[1] else ''}{o[1]}. Start the response with: the {o[0]}" if o[0] != o[1]
else f"Describe the relationship between the {o[0]} and the {'other ' if o[0]==o[1] else ''}{o[1]}. Start the response with: the {o[0]}"
for o in object_edges]

In [100]:
generated_texts = relationships
predicates = []
objects = []
results_relationships = []
results = [generated_texts[i].rstrip().split(':')[-1].lstrip() for i in range(len(generated_texts))]
results = [result if result != 'No relationship' else 'none' for result in results]
results = np.array(results)
results = results.tolist()
results_pred = []
for i, (r, objs) in enumerate(zip(results, object_edges)):
    if not (objs[0] in r and objs[1] in r):
        results_pred.append('none')
    else:
        try:
            results_pred.append(re.search(f'{objs[0]}(.*){objs[1]}', r).group(1).replace('the ', ''))
        except:
            results_pred.append('none')
objects = object_edges
results_relationships.extend(results)
predicates.extend(results_pred)

dist = torch.tensor(data_dict["predicate_min_dist"][0])
dist_mask = torch.norm(dist, dim=-1) > 0.5
predicates = np.array(predicates)
predicates[dist_mask] = 'none'

data_dict['edges'] = torch.tensor(data_dict['edges']).unsqueeze(0)
predicate_count = data_dict['predicate_count'].item()

objects_gt = []
for i in torch.unique(data_dict['edges']):
    mask = (data_dict['edges'] == i).cpu()[0][:predicate_count]
    objects_gt.append(objects[mask][0])

In [101]:
vis_graph_scannet_clip(data_dict['scan_id'], objects_gt, predicates, data_dict['edges'][0][:len(predicates)], data_dict['objects_id'], filename='graphs/'+scan_id)